# 🚗 Used Car Price Dataset - Data Cleaning & Exploratory Data Analysis (EDA)

This notebook performs an end-to-end data cleaning, anomaly detection, missing value imputation, feature engineering, and exploratory data analysis on the raw used car price dataset.

### Table of Contents:
1. **Environment Setup & Data Loading**
2. **Initial Data Profiling & Missing Value Analysis**
3. **Step-by-Step Data Cleaning Pipeline**
   - Dropping non-predictive `ID`
   - Duplicate removal
   - Target variable (`Price`) anomaly filtering
   - `Mileage` parsing and outlier treatment
   - `Engine volume` parsing & `Turbo` indicator extraction
   - `Doors` category standardization
   - Strategic imputation of missing values (median & mode)
   - Feature engineering (`Car_Age`)
4. **Validation & Cleaned Dataset Export (`car_price_cleaned.csv`)**
5. **Exploratory Data Analysis (EDA) & Visualizations**
   - Car Price Distribution & Metrics
   - Top 12 Most Common Car Manufacturers
   - Median Price by Vehicle Body Category
   - Vehicle Depreciation Curve (Price vs Car Age)
   - Correlation Heatmap of Numerical Features
   - Fuel Type Market Share & Price Spread


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'Arial', 'figure.dpi': 120})
%matplotlib inline

print("Libraries imported successfully!")


## 1. Load the Raw Dataset
Let's inspect the raw dataset `car_price.csv` to understand its schema, shape, and initial records.


In [ ]:
raw_file = "car_price.csv"
df_raw = pd.read_csv(raw_file)
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()


## 2. Initial Data Inspection & Missing Values
Let's inspect the data types and analyze missing values across each column.


In [ ]:
df_raw.info()


In [ ]:
# Missing values summary table
missing_df = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)


## 3. Data Cleaning Pipeline
We implement the comprehensive cleaning workflow:
1. **Drop `ID`**: It is an arbitrary identifier and offers no predictive signal.
2. **Drop Duplicates**: Remove duplicate rows.
3. **Target Variable (`Price`)**: Drop rows where `Price` is missing, and filter out extreme anomalies (< $500 dummy listings and > $150,000 errors).
4. **Clean `Mileage`**: Remove `' km'`, cast to numeric, and filter out corrupt values (> 600,000 km).
5. **Clean `Engine volume`**: Extract numeric displacement and create a binary `Turbo` feature.
6. **Standardize `Doors`**: Fix auto-converted Excel date representations (`04-May` -> `4-5`, etc.).
7. **Impute Missing Values**: Numerical features using median; categorical features using mode.
8. **Feature Engineering**: Compute `Car_Age` ($	ext{2020} - 	ext{Prod. year}$).


In [ ]:
df = df_raw.copy()

# 1. Drop non-predictive ID column
if "ID" in df.columns:
    df = df.drop(columns=["ID"])
    print("Dropped 'ID' column.")

# 2. Drop duplicate rows
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows. Remaining: {len(df):,}")

# 3. Handle Target: Price
df = df.dropna(subset=["Price"])
before_price = len(df)
df = df[(df["Price"] >= 500) & (df["Price"] <= 150000)]
print(f"Filtered {before_price - len(df)} price anomalies (< $500 or > $150,000). Remaining: {len(df):,}")

# 4. Clean Mileage
df["Mileage_km"] = df["Mileage"].astype(str).str.replace(" km", "", regex=False)
df["Mileage_km"] = pd.to_numeric(df["Mileage_km"], errors="coerce")
before_mileage = len(df)
df = df[(df["Mileage_km"].isnull()) | (df["Mileage_km"] <= 600000)]
print(f"Filtered {before_mileage - len(df)} unrealistic mileage records (> 600,000 km). Remaining: {len(df):,}")

# 5. Clean Engine volume & Extract Turbo
df["Turbo"] = df["Engine volume"].astype(str).str.contains("Turbo", case=False, na=False).astype(int)
df["Engine_Volume"] = df["Engine volume"].astype(str).str.replace(" Turbo", "", regex=False)
df["Engine_Volume"] = pd.to_numeric(df["Engine_Volume"], errors="coerce")
print("Extracted Engine_Volume and Turbo features.")

# 6. Clean Doors
door_mapping = {"04-May": "4-5", "02-Mar": "2-3", ">5": ">5"}
df["Doors"] = df["Doors"].map(door_mapping).fillna(df["Doors"])

# 7. Clean Leather interior
if "Leather interior" in df.columns:
    df["Leather interior"] = df["Leather interior"].map({"Yes": "Yes", "No": "No"}).fillna("No")

# 8. Imputation of numerical columns with median
num_imputations = {
    "Prod. year": df["Prod. year"].median(),
    "Mileage_km": df["Mileage_km"].median(),
    "Engine_Volume": df["Engine_Volume"].median(),
    "Levy": df["Levy"].median(),
    "Cylinders": df["Cylinders"].median(),
    "Airbags": df["Airbags"].median()
}
for col, val in num_imputations.items():
    if col in df.columns:
        df[col] = df[col].fillna(val)

# Imputation of categorical columns with mode
cat_cols = ["Manufacturer", "Model", "Category", "Fuel type", "Gear box type", "Drive wheels", "Doors", "Wheel", "Color"]
for col in cat_cols:
    if col in df.columns:
        mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else "Other"
        df[col] = df[col].fillna(mode_val)

# 9. Feature Engineering: Car_Age
current_year = int(df["Prod. year"].max())
df["Car_Age"] = (current_year - df["Prod. year"]).clip(lower=0).astype(int)

# Drop intermediate raw string columns and standardize naming
df = df.drop(columns=[c for c in ["Engine volume", "Mileage"] if c in df.columns])
df = df.rename(columns={
    "Prod. year": "Prod_Year",
    "Leather interior": "Leather_Interior",
    "Fuel type": "Fuel_Type",
    "Gear box type": "Gear_Box_Type",
    "Drive wheels": "Drive_Wheels"
})

print(f"\nCleaning Complete! Final Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")


## 4. Verification and Cleaned CSV Export
Let's verify that there are zero missing values left and save the final dataset to `car_price_cleaned.csv`.


In [ ]:
# Check for any remaining null values
print(f"Total null values in dataset: {df.isnull().sum().sum()}")

# Export cleaned CSV
cleaned_file = "car_price_cleaned.csv"
df.to_csv(cleaned_file, index=False)
print(f"Saved cleaned dataset to '{cleaned_file}' successfully!")
df.head()


## 5. Exploratory Data Analysis (EDA) & Visualizations


### Plot 1: Distribution of Car Prices
Understanding the spread, skewness, median, and mean of vehicle resale prices.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["Price"], kde=True, bins=50, color="#2563EB")
plt.axvline(df["Price"].median(), color="red", linestyle="--", label=f"Median: ${df['Price'].median():,.0f}")
plt.axvline(df["Price"].mean(), color="orange", linestyle=":", label=f"Mean: ${df['Price'].mean():,.0f}")
plt.title("Distribution of Car Prices (USD)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


### Plot 2: Top 12 Most Common Car Manufacturers
Analyzing market share across popular automobile brands in the dataset.


In [ ]:
plt.figure(figsize=(10, 5))
top_mfg = df["Manufacturer"].value_counts().nlargest(12)
sns.barplot(x=top_mfg.values, y=top_mfg.index, hue=top_mfg.index, palette="viridis", legend=False)
plt.title("Top 12 Most Common Car Manufacturers", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Number of Cars Listed")
plt.ylabel("Manufacturer")
for i, v in enumerate(top_mfg.values):
    plt.text(v + 15, i, f"{v:,}", va="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()


### Plot 3: Median Car Price by Vehicle Category
Comparing how different vehicle body types (Jeep, Sedan, Hatchback, etc.) retain their value.


In [ ]:
plt.figure(figsize=(10, 5))
cat_price = df.groupby("Category")["Price"].median().sort_values(ascending=False)
sns.barplot(x=cat_price.values, y=cat_price.index, hue=cat_price.index, palette="mako", legend=False)
plt.title("Median Car Price by Vehicle Category", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Median Price ($)")
plt.ylabel("Category")
for i, v in enumerate(cat_price.values):
    plt.text(v + 200, i, f"${v:,.0f}", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()


### Plot 4: Depreciation Curve (Median Price vs Car Age)
Demonstrating how car market values decline as vehicles age.


In [ ]:
plt.figure(figsize=(10, 5))
age_price = df[df["Car_Age"] <= 25].groupby("Car_Age")["Price"].median().reset_index()
sns.lineplot(data=age_price, x="Car_Age", y="Price", marker="o", color="#DC2626", linewidth=2.5)
plt.title("Median Car Price vs. Car Age (Depreciation Curve)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Car Age (Years)")
plt.ylabel("Median Price ($)")
plt.tight_layout()
plt.show()


### Plot 5: Correlation Heatmap of Numerical Features
Evaluating collinearity and linear relationships among numerical features and the target `Price`.


In [ ]:
plt.figure(figsize=(9, 6))
num_cols = ["Price", "Levy", "Prod_Year", "Car_Age", "Engine_Volume", "Mileage_km", "Cylinders", "Airbags", "Turbo"]
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True, linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


### Plot 6: Fuel Type Market Share & Price Spread
Analyzing the distribution of fuel types and their corresponding price spreads.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fuel_counts = df["Fuel_Type"].value_counts()
axes[0].pie(fuel_counts.values, labels=fuel_counts.index, autopct="%1.1f%%", startangle=140, colors=sns.color_palette("pastel"))
axes[0].set_title("Fuel Type Market Share", fontsize=12, fontweight="bold")

sns.boxplot(data=df, x="Fuel_Type", y="Price", ax=axes[1], hue="Fuel_Type", palette="Set2", legend=False, showfliers=False)
axes[1].set_title("Price Distribution by Fuel Type (Without Outliers)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Fuel Type")
axes[1].set_ylabel("Price ($)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 6. Key Takeaways & Next Steps
- **16,872 clean records** ready for machine learning.
- Strongest valuation drivers identified: `Car_Age`, `Mileage_km`, `Engine_Volume`, and `Category`.
- Next step: Run **`train_models.ipynb`** to train and benchmark 5 regression models.
